In [6]:
import requests
import pandas as pd
import time
import random
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
GITHUB_TOKEN = "Your Github Token"
BASE_URL = "https://api.github.com/search/repositories"

HEADERS = {
    "Accept": "application/vnd.github+json",
    "User-Agent": "python-requests",
    "Authorization": f"Bearer {GITHUB_TOKEN}"
}

PARAMS = {
    "q": "stars:>500",
    "sort": "stars",
    "order": "desc",
    "per_page": 100
}

# Setup session with retries
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

all_repos = []

# ✅ Efficient contributors count using pagination
def get_contributors_count(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/contributors"
    
    try:
        r = session.get(url, headers=HEADERS, params={"per_page": 1, "anon": "true"}, timeout=10)
        
        if r.status_code != 200:
            return None

        link = r.headers.get("Link", "")

        # Extract last page number
        if 'rel="last"' in link:
            last_page = link.split('rel="last"')[0].split("&page=")[-1].split(">")[0]
            return int(last_page)
        else:
            data = r.json()
            return len(data) if isinstance(data, list) else 0

    except requests.exceptions.RequestException:
        return None


# Fetch 500 repos
for page in range(1, 6):
    print(f"\nFetching page {page}...")
    PARAMS["page"] = page

    try:
        response = session.get(BASE_URL, headers=HEADERS, params=PARAMS, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("Search API failed:", e)
        continue

    print("Remaining requests:", response.headers.get("X-RateLimit-Remaining"))

    data = response.json()

    for i, repo in enumerate(data.get("items", []), start=1):
        owner = repo.get("owner", {}).get("login")
        name = repo.get("name")

        time.sleep(random.uniform(0.8, 1.5))
        contributors = get_contributors_count(owner, name)

        all_repos.append({
            "repo_name": name,
            "owner": owner,
            "language": repo.get("language"),
            "stars": repo.get("stargazers_count"),
            "forks": repo.get("forks_count"),
            "watchers": repo.get("watchers_count"),
            "open_issues": repo.get("open_issues_count"),
            "created_at": repo.get("created_at"),
            "updated_at": repo.get("updated_at"),
            "contributors": contributors,
            "topics": ",".join(repo.get("topics", [])),
            "description": repo.get("description"),
            "repo_url": repo.get("html_url")
        })

        time.sleep(random.uniform(0.5, 1.0))


# Save to CSV
df = pd.DataFrame(all_repos)
df.to_csv("github_repos_dataset.csv", index=False)

print(f"\nSaved {len(df)} repositories!")


Fetching page 1...
Remaining requests: 29
Processing codecrafters-io/build-your-own-x (1/100)
Processing sindresorhus/awesome (2/100)
Processing freeCodeCamp/freeCodeCamp (3/100)
Processing public-apis/public-apis (4/100)
Processing EbookFoundation/free-programming-books (5/100)
Processing openclaw/openclaw (6/100)
Processing kamranahmedse/developer-roadmap (7/100)
Processing donnemartin/system-design-primer (8/100)
Processing jwasham/coding-interview-university (9/100)
Processing vinta/awesome-python (10/100)
Processing awesome-selfhosted/awesome-selfhosted (11/100)
Processing 996icu/996.ICU (12/100)
Processing practical-tutorials/project-based-learning (13/100)
Processing facebook/react (14/100)
Processing torvalds/linux (15/100)
Processing TheAlgorithms/Python (16/100)
Processing trimstray/the-book-of-secret-knowledge (17/100)
Processing vuejs/vue (18/100)
Processing ossu/computer-science (19/100)
Processing trekhleb/javascript-algorithms (20/100)
Processing tensorflow/tensorflow (